In [1]:

# Upload a file from your local computer
from google.colab import files
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Saving heards_structured_cleaned_05.csv to heards_structured_cleaned_05.csv
User uploaded file "heards_structured_cleaned_05.csv" with length 42104713 bytes


In [3]:

import pandas as pd
import io

# Assuming the uploaded file is a CSV. Adjust the read function if it's different.
df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))

# Calculate the percentage of non-null values for each column
column_fill_percentage = (df.count() / len(df)) * 100

# Sort the results in decreasing order
sorted_column_fill_percentage = column_fill_percentage.sort_values(ascending=False)

sorted_column_fill_percentage

,0
updatedDate,100.000000
action_type,100.000000
price,100.000000
headline,100.000000
party,91.151008
end_date,91.102968
start_date,89.381209
grade,89.180874
location,82.739482
incoterm,82.203892


In [4]:
import pandas as pd

# Load your dataset (uploaded manually to Colab)
df = pd.read_csv('heards_structured_cleaned_05.csv')

# Convert date columns to datetime
df['updatedDate'] = pd.to_datetime(df['updatedDate'], errors='coerce')
df['start_date'] = pd.to_datetime(df['start_date'], format='%m-%d', errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], format='%m-%d', errors='coerce')

df.head()


,updatedDate,start_date,end_date,action_type,party,price,price_basis,grade,incoterm,location,volume_min,volume_max,frequency_tag,headline
0,2025-05-23 14:44:56.674,NaT,NaT,Offer,GLTD,16.0,3,NaN,CIF,Malta,NaN,NaN,Any Day,"Platts HSFO Med Crg CIF bss Malta 10-25, GLTD ..."
1,2025-05-23 08:28:59.221,NaT,NaT,Raise,TRAFI,2.5,MOPS 380,380.0,FOB,Straits,20.0,20.0,BalMnth,"Platts HSFO 380cst FOB Straits 15-30, TRAFI ra..."
2,2025-05-23 08:27:41.104,NaT,NaT,Raise,TRAFI,2.0,MOPS 380,380.0,FOB,Straits,20.0,20.0,BalMnth,"Platts HSFO 380cst FOB Straits 15-30, TRAFI ra..."
3,2025-05-23 08:26:22.775,NaT,NaT,Raise,TRAFI,1.5,MOPS 380,380.0,FOB,Straits,20.0,20.0,BalMnth,"Platts HSFO 380cst FOB Straits 15-30, TRAFI ra..."
4,2025-05-23 08:23:43.738,NaT,NaT,Raise,TRAFI,1.0,MOPS 380,380.0,FOB,Straits,20.0,20.0,BalMnth,"Platts HSFO 380cst FOB Straits 15-30, TRAFI ra..."


In [8]:
# Ensure datetime conversion for all 3 columns
df['updatedDate'] = pd.to_datetime(df['updatedDate'], errors='coerce')
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')

# Count nulls
total_rows = len(df)
null_updated = df['updatedDate'].isna().sum()
null_start = df['start_date'].isna().sum()
null_end = df['end_date'].isna().sum()

print(f"Total rows: {total_rows}")
print(f"Null updatedDate: {null_updated}")
print(f"Null start_date: {null_start}")
print(f"Null end_date: {null_end}")


Total rows: 195672
Null updatedDate: 74258
Null start_date: 20778
Null end_date: 17409


In [9]:
# Fix updatedDate parsing by enabling format inference
df['updatedDate'] = pd.to_datetime(df['updatedDate'], infer_datetime_format=True, errors='coerce')

# Recheck nulls
print("Null updatedDate after re-parsing:", df['updatedDate'].isna().sum())


Null updatedDate after re-parsing: 74258


<ipython-input-9-a50b4a3b81b4>:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['updatedDate'] = pd.to_datetime(df['updatedDate'], infer_datetime_format=True, errors='coerce')


In [10]:
# Reload raw updatedDate as string just in case it's partially parsed
df_raw = pd.read_csv('heards_structured_cleaned_05.csv')

# Extract only the date portion (first 10 characters) and parse it
df_raw['updatedDate_clean'] = pd.to_datetime(df_raw['updatedDate'].str.slice(0, 10), errors='coerce')

# Count how many are now null
null_count = df_raw['updatedDate_clean'].isna().sum()
print("Nulls after cleaning:", null_count)

# Replace original updatedDate column
df_raw['updatedDate'] = df_raw['updatedDate_clean']
df_raw.drop(columns='updatedDate_clean', inplace=True)


Nulls after cleaning: 0


In [12]:
# Save to a new file with cleaned updatedDate
df_raw.to_csv('heards_structured_cleaned_with_updatedDate.csv', index=False)

files.download('heards_structured_cleaned_with_updatedDate.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv('heards_structured_cleaned_with_updatedDate.csv')

# Parse date fields
df['updatedDate'] = pd.to_datetime(df['updatedDate'], errors='coerce')
df['start_date'] = pd.to_datetime(df['start_date'], format='%m-%d', errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], format='%m-%d', errors='coerce')

# Extract year from updatedDate
df['inferred_year'] = df['updatedDate'].dt.year

# Helper function to attach year to MM-DD dates
def add_year_to_date(month_day, year):
    if pd.isnull(month_day) or pd.isnull(year):
        return pd.NaT
    return month_day.replace(year=int(year))

# Apply to reconstruct full dates
df['inferred_start_date'] = df.apply(lambda row: add_year_to_date(row['start_date'], row['inferred_year']), axis=1)
df['inferred_end_date'] = df.apply(lambda row: add_year_to_date(row['end_date'], row['inferred_year']), axis=1)

# Print column fill rate and datatype
print("🧠 Column Fill Rates & Types")
summary = pd.DataFrame({
    'Non-Null Count': df.notna().sum(),
    'Total Rows': len(df),
    '% Filled': df.notna().mean() * 100,
    'Data Type': df.dtypes
}).sort_values('% Filled', ascending=False)

print(summary)


🧠 Column Fill Rates & Types
                     Non-Null Count  Total Rows    % Filled       Data Type
updatedDate                  195672      195672  100.000000  datetime64[ns]
action_type                  195672      195672  100.000000          object
price                        195672      195672  100.000000         float64
headline                     195672      195672  100.000000          object
inferred_year                195672      195672  100.000000           int32
party                        178357      195672   91.151008          object
grade                        174502      195672   89.180874         float64
location                     161898      195672   82.739482          object
incoterm                     160850      195672   82.203892          object
price_basis                  158335      195672   80.918578          object
volume_min                   119534      195672   61.088965         float64
volume_max                   119534      195672   61.088965 

In [16]:
# Show 5 random rows where start_date and end_date are NaT
sample_null_dates = df[df['start_date'].isna() | df['end_date'].isna()][
    ['updatedDate', 'start_date', 'end_date', 'headline']
].sample(5, random_state=42)

sample_null_dates


,updatedDate,start_date,end_date,headline
16261,2024-06-21,NaT,NaT,"Platts HSFO 380cst FOB Straits 15-30, TRAFI of..."
102523,2019-10-21,NaT,NaT,Asia 818: Platts HSFO 380cst FOB Straits 15-30...
155825,2018-02-27,NaT,NaT,Asia 260: Platts HSFO 380cst FOB Straits 15-30...
25738,2023-12-11,NaT,NaT,"Platts HSFO 380cst FOB Straits 15-30, TRAFI lo..."
184390,2021-12-11,NaT,NaT,platts singapore fuel oil bids offers trades


In [17]:
import re
from datetime import datetime

# Define month abbreviation to number mapping
month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

# Regex pattern to extract date ranges
pattern = r'(?P<start_month>[A-Za-z]{3})\s?(?P<start_day>\d{1,2})-(?:(?P<end_month>[A-Za-z]{3})\s?)?(?P<end_day>\d{1,2})'

# Function to extract and return full dates
def extract_inferred_dates(row):
    text = str(row['headline'])
    match = re.search(pattern, text)
    if match:
        year = pd.to_datetime(row['updatedDate']).year
        start_month = month_map.get(match.group('start_month'))
        start_day = int(match.group('start_day'))
        end_month = month_map.get(match.group('end_month') or match.group('start_month'))
        end_day = int(match.group('end_day'))

        try:
            inferred_start = datetime(year, start_month, start_day)
            inferred_end = datetime(year, end_month, end_day)
            return pd.Series([inferred_start, inferred_end])
        except:
            return pd.Series([pd.NaT, pd.NaT])
    return pd.Series([pd.NaT, pd.NaT])

# Apply function
df[['inferred_start_date', 'inferred_end_date']] = df.apply(extract_inferred_dates, axis=1)


In [18]:
df[['updatedDate', 'headline', 'inferred_start_date', 'inferred_end_date']].dropna().sample(5)


,updatedDate,headline,inferred_start_date,inferred_end_date
20165,2024-04-03,"Platts HSFO 380cst FOB ID 15-30, TRAFI lowers ...",2024-04-28,2024-05-02
15644,2024-07-04,Platts FUJ HSFO 380CST: Aug 8-Aug 13 pegged at...,2024-08-08,2024-08-13
14545,2024-08-01,Platts HSFO 180CST: Aug 21-Aug 25 pegged at MO...,2024-08-21,2024-08-25
42633,2022-10-14,"Platts HSFO 380cst FOB ID 15-30, VITOLSG lower...",2022-11-08,2022-11-12
1956,2025-04-23,Platts HSFO 380CST: May 18-May 23 pegged at MO...,2025-05-18,2025-05-23


In [20]:
import pandas as pd

# Truncate time from inferred dates
df['inferred_start_date'] = pd.to_datetime(df['inferred_start_date']).dt.date
df['inferred_end_date'] = pd.to_datetime(df['inferred_end_date']).dt.date

# Check fill rate
total_rows = len(df)
column_stats = []

for col in df.columns:
    non_null = df[col].notnull().sum()
    fill_percent = round((non_null / total_rows) * 100, 2)
    dtype = df[col].dtype
    column_stats.append((col, non_null, total_rows, fill_percent, dtype))

# Create summary dataframe
fill_summary = pd.DataFrame(
    column_stats,
    columns=["Column", "Non-Null Count", "Total Rows", "% Filled", "Data Type"]
)

# Sort by fill percentage
fill_summary = fill_summary.sort_values(by="% Filled", ascending=False).reset_index(drop=True)

# Display
print(fill_summary)


                 Column  Non-Null Count  Total Rows  % Filled       Data Type
0           updatedDate          195672      195672    100.00  datetime64[ns]
1           action_type          195672      195672    100.00          object
2                 price          195672      195672    100.00         float64
3              headline          195672      195672    100.00          object
4         inferred_year          195672      195672    100.00           int32
5                 party          178357      195672     91.15          object
6                 grade          174502      195672     89.18         float64
7              location          161898      195672     82.74          object
8              incoterm          160850      195672     82.20          object
9           price_basis          158335      195672     80.92          object
10           volume_min          119534      195672     61.09         float64
11           volume_max          119534      195672     61.09   

In [22]:
import pandas as pd
import re
from datetime import datetime

# Load the dataset
df = pd.read_csv("heards_structured_cleaned_with_updatedDate.csv")
df['updatedDate'] = pd.to_datetime(df['updatedDate'], errors='coerce')

# Function to extract dates from headline
def extract_dates(headline, updated):
    try:
        year = updated.year
        patterns = [
            r'(?P<start_month>[A-Za-z]+)\s(?P<start_day>\d{1,2})[-–](?P<end_month>[A-Za-z]+)?\s?(?P<end_day>\d{1,2})',  # Apr 28-May 2 or Apr 28-30
            r'(?P<start_month>[A-Za-z]+)\s(?P<start_day>\d{1,2})\s*to\s*(?P<end_month>[A-Za-z]+)?\s?(?P<end_day>\d{1,2})',  # July 8 to August 10
            r'valid\s(?P<start_month>[A-Za-z]+)\s(?P<start_day>\d{1,2})\s*[-–]\s*(?P<end_day>\d{1,2})',  # valid July 15 - 20
            r'(?P<start_month>\d{1,2})/(?P<start_day>\d{1,2})\s*[-–]\s*(?P<end_month>\d{1,2})/(?P<end_day>\d{1,2})'  # 10/15 - 10/30
        ]

        for pattern in patterns:
            match = re.search(pattern, headline)
            if match:
                gd = match.groupdict()

                if pattern.startswith('(?P<start_month>[A-Za-z]+)'):
                    start_month = gd['start_month']
                    end_month = gd.get('end_month') or start_month
                    start_date = pd.to_datetime(f"{start_month} {gd['start_day']} {year}", errors='coerce')
                    end_date = pd.to_datetime(f"{end_month} {gd['end_day']} {year}", errors='coerce')
                elif pattern.startswith('valid'):
                    start_month = gd['start_month']
                    start_date = pd.to_datetime(f"{start_month} {gd['start_day']} {year}", errors='coerce')
                    end_date = pd.to_datetime(f"{start_month} {gd['end_day']} {year}", errors='coerce')
                else:  # numeric month format
                    start_date = pd.to_datetime(f"{year}-{gd['start_month']}-{gd['start_day']}", errors='coerce')
                    end_date = pd.to_datetime(f"{year}-{gd['end_month']}-{gd['end_day']}", errors='coerce')

                return pd.Series([start_date, end_date])
    except:
        pass
    return pd.Series([pd.NaT, pd.NaT])

# Apply function
df[['inferred_start_date', 'inferred_end_date']] = df.apply(
    lambda row: extract_dates(row['headline'], row['updatedDate']), axis=1
)

# Convert inferred dates to datetime
df['inferred_start_date'] = pd.to_datetime(df['inferred_start_date'], errors='coerce')
df['inferred_end_date'] = pd.to_datetime(df['inferred_end_date'], errors='coerce')

# Print fill rate
fill_rate = df[['inferred_start_date', 'inferred_end_date']].notna().mean() * 100
print("🧠 Fill Rate (%):\n", fill_rate)

# Save if needed
df.to_csv("heards_structured_with_expanded_dates.csv", index=False)


🧠 Fill Rate (%):
 inferred_start_date    1.402347
inferred_end_date      1.402347
dtype: float64


In [23]:
import pandas as pd

# Truncate time from inferred dates
df['inferred_start_date'] = pd.to_datetime(df['inferred_start_date']).dt.date
df['inferred_end_date'] = pd.to_datetime(df['inferred_end_date']).dt.date

# Check fill rate
total_rows = len(df)
column_stats = []

for col in df.columns:
    non_null = df[col].notnull().sum()
    fill_percent = round((non_null / total_rows) * 100, 2)
    dtype = df[col].dtype
    column_stats.append((col, non_null, total_rows, fill_percent, dtype))

# Create summary dataframe
fill_summary = pd.DataFrame(
    column_stats,
    columns=["Column", "Non-Null Count", "Total Rows", "% Filled", "Data Type"]
)

# Sort by fill percentage
fill_summary = fill_summary.sort_values(by="% Filled", ascending=False).reset_index(drop=True)

# Display
print(fill_summary)


                 Column  Non-Null Count  Total Rows  % Filled       Data Type
0           updatedDate          195672      195672    100.00  datetime64[ns]
1           action_type          195672      195672    100.00          object
2                 price          195672      195672    100.00         float64
3              headline          195672      195672    100.00          object
4                 party          178357      195672     91.15          object
5              end_date          178263      195672     91.10          object
6            start_date          174894      195672     89.38          object
7                 grade          174502      195672     89.18         float64
8              location          161898      195672     82.74          object
9              incoterm          160850      195672     82.20          object
10          price_basis          158335      195672     80.92          object
11           volume_min          119534      195672     61.09   

In [24]:

!pip install python-dotenv

In [26]:

from google.colab import files
import io
from dotenv import load_dotenv
import os

# Upload the .env file
uploaded = files.upload()

# Assuming the uploaded file is '.env'
if '.env' in uploaded:
    # Write the content to a file named .env in the current directory
    with open('.env', 'wb') as f:
        f.write(uploaded['.env'])
    print("'.env' file uploaded and saved.")

    # Load the environment variables from the .env file
    load_dotenv()

    # You can now access variables using os.getenv()
    # For example:
    # db_user = os.getenv('DB_USER')
    # db_password = os.getenv('DB_PASSWORD')
    # print(f"DB_USER: {db_user}")
    # print(f"DB_PASSWORD: {db_password}")

else:
    print("'.env' file was not uploaded.")


Saving .env to .env
'.env' file uploaded and saved.


In [27]:
from dotenv import load_dotenv
import os

# Upload .env file manually in the left panel
load_dotenv('.env')
api_key = os.getenv("AI_API_KEY")

if not api_key:
    raise ValueError("API key not found in .env file.")


In [28]:
!pip install openai
import openai
openai.api_key = api_key


In [33]:
import pandas as pd

# Load full dataset
df = pd.read_csv("heards_structured_cleaned_with_updatedDate.csv", parse_dates=["updatedDate"])

# Reset start_date and end_date to null
df["start_date"] = pd.NaT
df["end_date"] = pd.NaT

# Recalculate fill rates
def compute_fill_rates(dataframe):
    total_rows = len(dataframe)
    summary = []
    for col in dataframe.columns:
        non_null = dataframe[col].notnull().sum()
        dtype = dataframe[col].dtype
        fill_pct = round(100 * non_null / total_rows, 2)
        summary.append([col, non_null, total_rows, fill_pct, dtype])
    summary_df = pd.DataFrame(summary, columns=["Column", "Non-Null Count", "Total Rows", "% Filled", "Data Type"])
    return summary_df.sort_values(by="% Filled", ascending=False)

# Print full dataset fill rates
print("🧠 Full Dataset Column Fill Rate (%):")
print(compute_fill_rates(df))

# Save a random 500-row sample (we'll label this)
sample_500 = df.sample(n=500, random_state=42)
sample_500.to_csv("heards_sample_500.csv", index=False)
print("\n✅ Saved 'heards_sample_500.csv' for manual/OpenAI labeling.")


🧠 Full Dataset Column Fill Rate (%):
           Column  Non-Null Count  Total Rows  % Filled       Data Type
0     updatedDate          195672      195672    100.00  datetime64[ns]
3     action_type          195672      195672    100.00          object
5           price          195672      195672    100.00         float64
13       headline          195672      195672    100.00          object
4           party          178357      195672     91.15          object
7           grade          174502      195672     89.18         float64
9        location          161898      195672     82.74          object
8        incoterm          160850      195672     82.20          object
6     price_basis          158335      195672     80.92          object
10     volume_min          119534      195672     61.09         float64
11     volume_max          119534      195672     61.09         float64
12  frequency_tag          118892      195672     60.76          object
2        end_date          

In [35]:
from google.colab import files

# Download the sample file
files.download("heards_sample_500.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
# Step 1: Upload your .env file manually in the left pane of Colab

# Step 2: Install dotenv and load your key
!pip install python-dotenv
from dotenv import load_dotenv
import os

load_dotenv(".env")  # Ensure the .env file is uploaded and this path is correct
openai.api_key = os.getenv("AI_API_KEY")

# Step 3: Confirm it's working (optional but safe)
if openai.api_key is None:
    raise ValueError("API key not loaded. Check your .env file.")

# Step 4: Rerun the GPT batch call script once your key is loaded


In [50]:
import pandas as pd
import openai
import time

# Load sample data
df_sample = pd.read_csv("heards_sample_500.csv")
df_sample['updatedDate'] = pd.to_datetime(df_sample['updatedDate'])

# Define function to extract start and end dates from GPT
def extract_dates_with_openai(row):
    prompt = f"""
You are given the following:
- Updated date: {row['updatedDate'].strftime('%Y-%m-%d')}
- Headline: "{row['headline']}"

Your job is to extract the **start date** and **end date** (if they exist) from the headline.
Assume missing year values are the same as the updated year unless it would logically fall in the next/previous year.
Output format must be:
start_date: YYYY-MM-DD
end_date: YYYY-MM-DD

If either is missing, write: null
Return only the two fields exactly as instructed.
    """

    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You extract start and end dates from commodity trading headlines."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
        )
        return response.choices[0].message["content"].strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# Process rows with GPT
results = []
for idx, row in df_sample.iterrows():
    print(f"Processing row {idx+1}/{len(df_sample)}...")
    results.append(extract_dates_with_openai(row))


# Save results
df_sample["gpt_response"] = results
df_sample.to_csv("heards_sample_500_with_gpt.csv", index=False)
print("✅ Saved as 'heards_sample_500_with_gpt.csv'")


Processing row 1/500...
Processing row 2/500...
Processing row 3/500...
Processing row 4/500...
Processing row 5/500...
Processing row 6/500...
Processing row 7/500...
Processing row 8/500...
Processing row 9/500...
Processing row 10/500...
Processing row 11/500...
Processing row 12/500...
Processing row 13/500...
Processing row 14/500...
Processing row 15/500...
Processing row 16/500...
Processing row 17/500...
Processing row 18/500...
Processing row 19/500...
Processing row 20/500...
Processing row 21/500...
Processing row 22/500...
Processing row 23/500...
Processing row 24/500...
Processing row 25/500...
Processing row 26/500...
Processing row 27/500...
Processing row 28/500...
Processing row 29/500...
Processing row 30/500...
Processing row 31/500...
Processing row 32/500...
Processing row 33/500...
Processing row 34/500...
Processing row 35/500...
Processing row 36/500...
Processing row 37/500...
Processing row 38/500...
Processing row 39/500...
Processing row 40/500...
Processin

In [51]:
df_sample["gpt_response"] = results


In [52]:
df_sample.to_csv("heards_sample_500_labeled.csv", index=False)
print("✅ Saved GPT-labeled responses to 'heards_sample_500_labeled.csv'")


✅ Saved GPT-labeled responses to 'heards_sample_500_labeled.csv'


In [53]:
import re
import pandas as pd

# Load the labeled sample
df = pd.read_csv("heards_sample_500_labeled.csv")
df['updatedDate'] = pd.to_datetime(df['updatedDate'])

# Extract start and end dates using regex
def parse_dates(response):
    try:
        start = re.search(r"start_date:\s*(\d{4}-\d{2}-\d{2}|null)", response)
        end = re.search(r"end_date:\s*(\d{4}-\d{2}-\d{2}|null)", response)

        start_date = pd.to_datetime(start.group(1)) if start and start.group(1) != "null" else pd.NaT
        end_date = pd.to_datetime(end.group(1)) if end and end.group(1) != "null" else pd.NaT

        return pd.Series([start_date, end_date])
    except:
        return pd.Series([pd.NaT, pd.NaT])

# Apply parsing
df[["start_date", "end_date"]] = df["gpt_response"].apply(parse_dates)

# Save parsed result
df.to_csv("heards_sample_500_parsed.csv", index=False)
print("✅ Final parsed file saved as 'heards_sample_500_parsed.csv'")


✅ Final parsed file saved as 'heards_sample_500_parsed.csv'


In [54]:
from google.colab import files
files.download("heards_sample_500_parsed.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
# Install latest OpenAI SDK
!pip install --upgrade openai

# Import & load key from .env
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load .env and initialize client
load_dotenv(".env")
api_key = os.getenv("AI_API_KEY")

if api_key is None:
    raise ValueError("❌ API key not found. Check your .env file.")

# Initialize OpenAI client
client = OpenAI(api_key=api_key)

# Make a simple test call
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello in French."}
    ],
    temperature=0,
)

# Print response
print("✅ GPT Test Response:", response.choices[0].message.content)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.5/720.5 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.81.0
    Uninstalling openai-1.81.0:
      Successfully uninstalled openai-1.81.0
✅ GPT Test Response: Bonjour


In [56]:
import pandas as pd
import time
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(".env")
api_key = os.getenv("AI_API_KEY")
if api_key is None:
    raise ValueError("❌ API key not loaded. Check your .env file.")

# Load 500-row dataset
df_sample = pd.read_csv("heards_sample_500.csv")
df_sample['updatedDate'] = pd.to_datetime(df_sample['updatedDate'])

# OpenAI client
client = OpenAI(api_key=api_key)

# Helper function to call GPT-4 and return JSON output
def call_gpt_json(row):
    prompt = f"""
You are given the following:
- Updated date: {row['updatedDate'].strftime('%Y-%m-%d')}
- Headline: "{row['headline']}"

Your task is to extract the **start_date** and **end_date** from the headline, if available.

Assume:
- If no year is mentioned, use the year from the updatedDate (unless another year is clearly implied).
- Dates may span months or be in formats like "Nov 5-Nov 9", "Mar 21-Mar 25", etc.

Respond **only** in this exact JSON format:

{{
  "start_date": "YYYY-MM-DD" or null,
  "end_date": "YYYY-MM-DD" or null
}}

Do not include any other explanation or text.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You extract structured dates from commodity trading headlines."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# Run on 500 rows
results = []
for idx, row in df_sample.iterrows():
    print(f"🔄 Processing row {idx+1}/500")
    result = call_gpt_json(row)
    results.append(result)


# Save raw JSON response
df_sample["gpt_response"] = results
df_sample.to_csv("heards_sample_500_raw_response.csv", index=False)
print("✅ Responses saved to heards_sample_500_raw_response.csv")


🔄 Processing row 1/500
🔄 Processing row 2/500
🔄 Processing row 3/500
🔄 Processing row 4/500
🔄 Processing row 5/500
🔄 Processing row 6/500
🔄 Processing row 7/500
🔄 Processing row 8/500
🔄 Processing row 9/500
🔄 Processing row 10/500
🔄 Processing row 11/500
🔄 Processing row 12/500
🔄 Processing row 13/500
🔄 Processing row 14/500
🔄 Processing row 15/500
🔄 Processing row 16/500
🔄 Processing row 17/500
🔄 Processing row 18/500
🔄 Processing row 19/500
🔄 Processing row 20/500
🔄 Processing row 21/500
🔄 Processing row 22/500
🔄 Processing row 23/500
🔄 Processing row 24/500
🔄 Processing row 25/500
🔄 Processing row 26/500
🔄 Processing row 27/500
🔄 Processing row 28/500
🔄 Processing row 29/500
🔄 Processing row 30/500
🔄 Processing row 31/500
🔄 Processing row 32/500
🔄 Processing row 33/500
🔄 Processing row 34/500
🔄 Processing row 35/500
🔄 Processing row 36/500
🔄 Processing row 37/500
🔄 Processing row 38/500
🔄 Processing row 39/500
🔄 Processing row 40/500
🔄 Processing row 41/500
🔄 Processing row 42/500
🔄

In [57]:
from google.colab import files
files.download("heards_sample_500_raw_response.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [58]:
import pandas as pd
import re

# Load the raw GPT responses
df = pd.read_csv("heards_sample_500_raw_response.csv")

# Define parser to extract dates in expected format
def parse_dates(text):
    start_match = re.search(r"start_date:\s*(\d{4}-\d{2}-\d{2}|null)", text)
    end_match = re.search(r"end_date:\s*(\d{4}-\d{2}-\d{2}|null)", text)

    start = start_match.group(1) if start_match else "null"
    end = end_match.group(1) if end_match else "null"

    return pd.Series([start if start != "null" else None,
                      end if end != "null" else None])

# Apply parser to gpt_response column
df[["parsed_start_date", "parsed_end_date"]] = df["gpt_response"].apply(parse_dates)

# Save parsed version
output_path = "heards_sample_500_parsed.csv"
df.to_csv(output_path, index=False)
print(f"✅ Cleaned file saved to: {output_path}")
from google.colab import files
files.download("heards_sample_500_parsed.csv")

✅ Cleaned file saved to: heards_sample_500_parsed.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [61]:
import pandas as pd

# Load parsed file
df = pd.read_csv("heards_sample_500_parsed.csv")

# Convert parsed date columns to datetime
df['parsed_start_date'] = pd.to_datetime(df['parsed_start_date'], errors='coerce')
df['parsed_end_date'] = pd.to_datetime(df['parsed_end_date'], errors='coerce')

# Check fill rate
start_fill = df['parsed_start_date'].notnull().mean() * 100
end_fill = df['parsed_end_date'].notnull().mean() * 100

print(f"✅ Parsed Start Date Fill Rate: {start_fill:.2f}%")
print(f"✅ Parsed End Date Fill Rate: {end_fill:.2f}%")

# Preview
df[['updatedDate', 'headline', 'parsed_start_date', 'parsed_end_date','gpt_response']].head(10)


✅ Parsed Start Date Fill Rate: 0.00%
✅ Parsed End Date Fill Rate: 0.00%


,updatedDate,headline,parsed_start_date,parsed_end_date,gpt_response
0,2024-06-21,"Platts HSFO 380cst FOB Straits 15-30, TRAFI of...",NaT,NaT,"{\n ""start_date"": ""2024-07-12"",\n ""end_date""..."
1,2019-10-21,Asia 818: Platts HSFO 380cst FOB Straits 15-30...,NaT,NaT,"{\n ""start_date"": ""2019-11-05"",\n ""end_date""..."
2,2018-02-27,Asia 260: Platts HSFO 380cst FOB Straits 15-30...,NaT,NaT,"{\n ""start_date"": ""2018-03-21"",\n ""end_date""..."
3,2023-12-11,"Platts HSFO 380cst FOB Straits 15-30, TRAFI lo...",NaT,NaT,"{\n ""start_date"": ""2023-12-26"",\n ""end_date""..."
4,2021-12-11,platts singapore fuel oil bids offers trades,NaT,NaT,"{\n ""start_date"": null,\n ""end_date"": null\n}"
5,2018-10-08,Asia 191: Platts HSFO 380cst FOB Straits 15-30...,NaT,NaT,"{\n ""start_date"": ""2018-10-27"",\n ""end_date""..."
6,2018-03-07,asia 2001: platts hsfo: physical bids finals o...,NaT,NaT,"{\n ""start_date"": ""2001-01-01"",\n ""end_date""..."
7,2024-09-17,"Platts HSFO 180cst FOB Straits 15-30, TRAFI ra...",NaT,NaT,"{\n ""start_date"": ""2024-10-07"",\n ""end_date""..."
8,2020-12-22,"Platts HSFO 180cst FOB Straits 15-30, VITOLSG ...",NaT,NaT,"{\n ""start_date"": ""2021-01-16"",\n ""end_date""..."
9,2017-10-04,Asia 1563: Platts HSFO 380cst FOB Straits 15-3...,NaT,NaT,"{\n ""start_date"": ""2017-10-24"",\n ""end_date""..."


In [62]:
import pandas as pd
import json

# Load CSV
df = pd.read_csv("heards_sample_500_raw_response.csv")

# Parse JSON responses into new columns
def parse_dates_from_json(response):
    try:
        parsed = json.loads(response)
        return pd.Series([parsed.get("start_date"), parsed.get("end_date")])
    except:
        return pd.Series([None, None])

# Apply parsing
df[['parsed_start_date', 'parsed_end_date']] = df['gpt_response'].apply(parse_dates_from_json)

# Convert to datetime
df['parsed_start_date'] = pd.to_datetime(df['parsed_start_date'], errors='coerce')
df['parsed_end_date'] = pd.to_datetime(df['parsed_end_date'], errors='coerce')

# Check fill rates
start_fill = df['parsed_start_date'].notnull().mean() * 100
end_fill = df['parsed_end_date'].notnull().mean() * 100

print(f"✅ Parsed Start Date Fill Rate: {start_fill:.2f}%")
print(f"✅ Parsed End Date Fill Rate: {end_fill:.2f}%")

# Save cleaned version
df.to_csv("heards_sample_500_parsed_fixed.csv", index=False)


✅ Parsed Start Date Fill Rate: 94.40%
✅ Parsed End Date Fill Rate: 94.00%


In [63]:
from google.colab import files
files.download("heards_sample_500_parsed_fixed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
import pandas as pd

df = pd.read_csv("heards_sample_500_parsed_fixed.csv")
print(df[['headline', 'parsed_start_date', 'parsed_end_date']].head())


                                            headline parsed_start_date  \
0  Platts HSFO 380cst FOB Straits 15-30, TRAFI of...        2024-07-12   
1  Asia 818: Platts HSFO 380cst FOB Straits 15-30...        2019-11-05   
2  Asia 260: Platts HSFO 380cst FOB Straits 15-30...        2018-03-21   
3  Platts HSFO 380cst FOB Straits 15-30, TRAFI lo...        2023-12-26   
4       platts singapore fuel oil bids offers trades               NaN   

  parsed_end_date  
0      2024-07-16  
1      2019-11-09  
2      2018-03-25  
3      2023-12-30  
4             NaN  


In [66]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from datetime import datetime

# Load data
df = pd.read_csv("heards_sample_500_parsed_fixed.csv")

# Drop rows without labels
df_model = df.dropna(subset=["parsed_start_date", "parsed_end_date"]).copy()

# Convert to datetime
df_model['parsed_start_date'] = pd.to_datetime(df_model['parsed_start_date'])
df_model['parsed_end_date'] = pd.to_datetime(df_model['parsed_end_date'])
df_model['updatedDate'] = pd.to_datetime(df_model['updatedDate'])

# Convert labels to ordinal (number of days)
df_model['start_label'] = (df_model['parsed_start_date'] - pd.Timestamp("1970-01-01")) // pd.Timedelta("1D")
df_model['end_label'] = (df_model['parsed_end_date'] - pd.Timestamp("1970-01-01")) // pd.Timedelta("1D")


In [67]:
X = df_model[['headline', 'updatedDate']]
y = df_model[['start_label', 'end_label']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [68]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Feature engineering: Convert date to number of days since epoch
def days_since_epoch(X):
    return np.array([(x - pd.Timestamp("1970-01-01")).days for x in X]).reshape(-1, 1)

date_transformer = FunctionTransformer(days_since_epoch, validate=False)
text_transformer = TfidfVectorizer(max_features=1000)

preprocessor = ColumnTransformer(transformers=[
    ('text', text_transformer, 'headline'),
    ('date', date_transformer, 'updatedDate')
])

# Full pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', MultiOutputRegressor(Ridge()))
])

# Train model
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  TfidfVectorizer(max_features=1000),
                                                  'headline'),
                                                 ('date',
                                                  FunctionTransformer(func=<function days_since_epoch at 0x7e3090a53420>),
                                                  'updatedDate')])),
                ('regressor', MultiOutputRegressor(estimator=Ridge()))])

In [69]:
from sklearn.metrics import mean_absolute_error

y_pred = pipeline.predict(X_test)

mae_start = mean_absolute_error(y_test['start_label'], y_pred[:, 0])
mae_end = mean_absolute_error(y_test['end_label'], y_pred[:, 1])

print(f"MAE (Start Date): {mae_start:.2f} days")
print(f"MAE (End Date): {mae_end:.2f} days")


MAE (Start Date): 23.81 days
MAE (End Date): 23.32 days


In [70]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import mean_absolute_error
from datetime import datetime

# Load the labeled dataset
df = pd.read_csv("heards_sample_500_parsed_fixed.csv")

# Drop rows with missing parsed_start_date or parsed_end_date
df = df.dropna(subset=["parsed_start_date", "parsed_end_date"])

# Parse dates
df["updatedDate"] = pd.to_datetime(df["updatedDate"])
df["parsed_start_date"] = pd.to_datetime(df["parsed_start_date"])
df["parsed_end_date"] = pd.to_datetime(df["parsed_end_date"])

# Compute the number of days from updatedDate to start and end
df["start_offset"] = (df["parsed_start_date"] - df["updatedDate"]).dt.days
df["end_offset"] = (df["parsed_end_date"] - df["updatedDate"]).dt.days

# Feature: headline text
X_text = df["headline"]
# Feature: days since epoch from updatedDate
X_date = df["updatedDate"].apply(lambda x: (x - pd.Timestamp("1970-01-01")) // pd.Timedelta('1D'))

# Combine headline TF-IDF with date as feature
vectorizer = TfidfVectorizer(max_features=1000)

X_text_vec = vectorizer.fit_transform(X_text)
X_full = np.hstack((X_text_vec.toarray(), X_date.values.reshape(-1, 1)))

# Targets
y_start = df["start_offset"].values
y_end = df["end_offset"].values

# Train/test split
X_train, X_test, y_start_train, y_start_test, y_end_train, y_end_test = train_test_split(
    X_full, y_start, y_end, test_size=0.2, random_state=42
)

# Train models
model_start = Ridge()
model_end = Ridge()

model_start.fit(X_train, y_start_train)
model_end.fit(X_train, y_end_train)

# Predict
y_start_pred = model_start.predict(X_test)
y_end_pred = model_end.predict(X_test)

# Evaluate
mae_start = mean_absolute_error(y_start_test, y_start_pred)
mae_end = mean_absolute_error(y_end_test, y_end_pred)

print(f"MAE (Start Date Offset): {mae_start:.2f} days")
print(f"MAE (End Date Offset): {mae_end:.2f} days")

# Predict on full set (optional)
df["predicted_start_date"] = df["updatedDate"] + pd.to_timedelta(model_start.predict(X_full).round(), unit="D")
df["predicted_end_date"] = df["updatedDate"] + pd.to_timedelta(model_end.predict(X_full).round(), unit="D")

# Save results
df.to_csv("heards_sample_500_predicted.csv", index=False)


MAE (Start Date Offset): 26.05 days
MAE (End Date Offset): 25.49 days


In [71]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from datetime import datetime

# === Load and preprocess ===
df = pd.read_csv("heards_sample_500_parsed_fixed.csv")
df = df.dropna(subset=["parsed_start_date", "parsed_end_date"])

# Convert dates
df["updatedDate"] = pd.to_datetime(df["updatedDate"])
df["parsed_start_date"] = pd.to_datetime(df["parsed_start_date"])
df["parsed_end_date"] = pd.to_datetime(df["parsed_end_date"])

# Create offset targets
df["start_offset"] = (df["parsed_start_date"] - df["updatedDate"]).dt.days
df["end_offset"] = (df["parsed_end_date"] - df["updatedDate"]).dt.days

# === Feature Engineering ===
df["day_of_week"] = df["updatedDate"].dt.dayofweek
df["month"] = df["updatedDate"].dt.month
df["day"] = df["updatedDate"].dt.day

# Text features (TF-IDF with n-grams)
tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=5000)
X_text = tfidf.fit_transform(df["headline"])

# Combine numeric and text features
X_numeric = df[["day_of_week", "month", "day"]].values
from scipy.sparse import hstack
X = hstack([X_text, X_numeric])

# === Train-Test Split ===
X_train, X_test, y_start_train, y_start_test = train_test_split(X, df["start_offset"], test_size=0.2, random_state=42)
_, _, y_end_train, y_end_test = train_test_split(X, df["end_offset"], test_size=0.2, random_state=42)

# === Model Training ===
start_model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
end_model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)

start_model.fit(X_train, y_start_train)
end_model.fit(X_train, y_end_train)

# === Predictions ===
y_start_pred = start_model.predict(X_test)
y_end_pred = end_model.predict(X_test)

# === Evaluation ===
start_mae = mean_absolute_error(y_start_test, y_start_pred)
end_mae = mean_absolute_error(y_end_test, y_end_pred)

print(f"📊 MAE (Start Offset): {start_mae:.2f} days")
print(f"📊 MAE (End Offset): {end_mae:.2f} days")


📊 MAE (Start Offset): 3.98 days
📊 MAE (End Offset): 4.03 days


In [79]:
import pandas as pd
import numpy as np

# === Load Full Dataset ===
df_full = pd.read_csv("heards_structured_cleaned_with_updatedDate.csv", parse_dates=["updatedDate"])

# === Preserve all original columns ===
df_original = df_full.copy()

# === Filter rows needed for model prediction ===
df_model_input = df_original[["updatedDate", "headline"]].dropna()

# === Transform Inputs ===
X_tfidf = vectorizer.transform(df_model_input["headline"])
X_date = df_model_input["updatedDate"].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
X_all = np.hstack([X_tfidf.toarray(), X_date])

# === Predict Offsets ===
pred_start_offset = model_start.predict(X_all).round().astype(int)
pred_end_offset = model_end.predict(X_all).round().astype(int)

# === Convert Offsets to Dates ===
pred_start_date = df_model_input["updatedDate"] + pd.to_timedelta(pred_start_offset, unit="D")
pred_end_date = df_model_input["updatedDate"] + pd.to_timedelta(pred_end_offset, unit="D")

# === Assign Predictions Back ===
df_original.loc[df_model_input.index, "start_date"] = pred_start_date
df_original.loc[df_model_input.index, "end_date"] = pred_end_date

# === Save Final File with All Columns ===
df_original.to_csv("heards_structured_with_predicted_dates2.csv", index=False)
print("✅ Final file saved: 'heards_structured_with_predicted_dates.csv'")


✅ Final file saved: 'heards_structured_with_predicted_dates.csv'


In [80]:

files.download("heards_structured_with_predicted_dates2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [81]:

# Load the file
df = pd.read_csv('heards_structured_with_predicted_dates2.csv')

# Get the total number of rows
total_rows = len(df)

# Calculate the percentage of non-null values for each column
column_fill_percentage = (df.count() / total_rows) * 100

# Create a summary DataFrame
column_stats = pd.DataFrame({
    'Total Rows': total_rows,
    'Non-Null Count': df.notna().sum(),
    '% Filled': column_fill_percentage,
    'Data Type': df.dtypes
})

# Sort by fill percentage
sorted_column_stats = column_stats.sort_values(by='% Filled', ascending=False)

# Display the results
print("\n📊 Column Fill Rate & Data Types for 'heards_fully_labeled.csv':")
sorted_column_stats



📊 Column Fill Rate & Data Types for 'heards_fully_labeled.csv':


,Total Rows,Non-Null Count,% Filled,Data Type
updatedDate,195672,195672,100.000000,object
start_date,195672,195672,100.000000,object
end_date,195672,195672,100.000000,object
action_type,195672,195672,100.000000,object
price,195672,195672,100.000000,float64
headline,195672,195672,100.000000,object
party,195672,178357,91.151008,object
grade,195672,174502,89.180874,float64
location,195672,161898,82.739482,object
incoterm,195672,160850,82.203892,object


In [84]:

# Load the specified file
df_predicted = pd.read_csv('heards_structured_with_predicted_dates2.csv')

# Select and print 5 random rows with the specified columns
print(df_predicted[['updatedDate', 'start_date', 'end_date', 'headline']].sample(5).to_markdown(index=False))

| updatedDate   | start_date          | end_date            | headline                                                                                                                 |
|:--------------|:--------------------|:--------------------|:-------------------------------------------------------------------------------------------------------------------------|
| 2018-03-09    | 2024-03-19 00:00:00 | 2022-07-16 00:00:00 | Asia 867: Platts HSFO 380cst FOB Straits 15-30, COASTAL raises bid Apr 4-Apr 8 100% MOPS 380 5 Day $-1.50  for 20-20 "IN |
| 2019-09-19    | 2025-08-25 00:00:00 | 2023-12-20 00:00:00 | Asia 601: Platts HSFO 380cst FOB Straits 15-30, GLENCORESG lowers offer Oct 4-Oct 8 100% MOPS 380 Full Mnth Oct $81.00   |
| 2017-11-06    | 2023-12-21 00:00:00 | 2022-04-08 00:00:00 | Asia 1332: Platts HSFO 380cst FOB Straits 15-30, MERCURIASG raises bid Nov 25-Nov 29 100% MOPS 380 5 Day $1.25  for 20-2 |
| 2018-06-05    | 2024-05-24 00:00:00 | 2022-09-15 00:00:00 | Asia 1367: Pl

In [85]:
# === SETUP ===
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# === UPLOAD CSV FILE ===
from google.colab import files
uploaded = files.upload()

# === LOAD & CLEAN DATA ===
df = pd.read_csv("heards_sample_500_parsed_fixed.csv", parse_dates=["updatedDate", "parsed_start_date", "parsed_end_date"])
df = df.dropna(subset=["headline", "parsed_start_date", "parsed_end_date", "updatedDate"])

# === CREATE TARGETS ===
df["start_month"] = df["parsed_start_date"].dt.month
df["start_day"] = df["parsed_start_date"].dt.day
df["end_month"] = df["parsed_end_date"].dt.month
df["end_day"] = df["parsed_end_date"].dt.day

# === VECTORIZE HEADLINES ===
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(df["headline"])

# === SPLIT TARGETS ===
y_sm = df["start_month"]
y_sd = df["start_day"]
y_em = df["end_month"]
y_ed = df["end_day"]

# === TRAIN/TEST SPLIT ===
X_train, X_test, y_sm_train, y_sm_test = train_test_split(X, y_sm, test_size=0.2, random_state=42)
_, _, y_sd_train, y_sd_test = train_test_split(X, y_sd, test_size=0.2, random_state=42)
_, _, y_em_train, y_em_test = train_test_split(X, y_em, test_size=0.2, random_state=42)
_, _, y_ed_train, y_ed_test = train_test_split(X, y_ed, test_size=0.2, random_state=42)

# === TRAIN MODELS ===
model_sm = RandomForestClassifier()
model_sd = RandomForestClassifier()
model_em = RandomForestClassifier()
model_ed = RandomForestClassifier()

model_sm.fit(X_train, y_sm_train)
model_sd.fit(X_train, y_sd_train)
model_em.fit(X_train, y_em_train)
model_ed.fit(X_train, y_ed_train)

# === EVALUATE ===
print("📅 Start Month Accuracy:", accuracy_score(y_sm_test, model_sm.predict(X_test)))
print("📅 Start Day Accuracy:", accuracy_score(y_sd_test, model_sd.predict(X_test)))
print("📅 End Month Accuracy:", accuracy_score(y_em_test, model_em.predict(X_test)))
print("📅 End Day Accuracy:", accuracy_score(y_ed_test, model_ed.predict(X_test)))


Saving heards_sample_500_parsed_fixed.csv to heards_sample_500_parsed_fixed (1).csv
📅 Start Month Accuracy: 0.9148936170212766
📅 Start Day Accuracy: 0.6702127659574468
📅 End Month Accuracy: 0.9680851063829787
📅 End Day Accuracy: 0.6808510638297872


In [86]:
import numpy as np

# Predict all date components using the trained models
pred_start_month = model_sm.predict(X)
pred_start_day = model_sd.predict(X)
pred_end_month = model_em.predict(X)
pred_end_day = model_ed.predict(X)

# Reconstruct full start and end dates (force year from updatedDate)
pred_start_dates = pd.to_datetime({
    "year": df["updatedDate"].dt.year,
    "month": pred_start_month,
    "day": pred_start_day
}, errors="coerce")  # Invalid dates become NaT

pred_end_dates = pd.to_datetime({
    "year": df["updatedDate"].dt.year,
    "month": pred_end_month,
    "day": pred_end_day
}, errors="coerce")

# Add predictions to DataFrame
df["predicted_start_date"] = pred_start_dates
df["predicted_end_date"] = pred_end_dates


In [87]:
# Identify rows where prediction failed (invalid date combinations like Feb 30)
invalid_start = df["predicted_start_date"].isna()
invalid_end = df["predicted_end_date"].isna()

# Combine flags
invalid_rows = invalid_start | invalid_end

# Summary
print("❌ Invalid predicted start dates:", invalid_start.sum())
print("❌ Invalid predicted end dates:", invalid_end.sum())
print("❌ Total invalid rows:", invalid_rows.sum())

# Optionally filter valid predictions only
df_valid_predictions = df[~invalid_rows]


❌ Invalid predicted start dates: 0
❌ Invalid predicted end dates: 0
❌ Total invalid rows: 0


In [88]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

# === STEP 1: Load labeled training data ===
df_train = pd.read_csv("heards_sample_500_parsed_fixed.csv", parse_dates=["updatedDate", "parsed_start_date", "parsed_end_date"])
df_train = df_train.dropna(subset=["headline", "parsed_start_date", "parsed_end_date", "updatedDate"]).copy()

# Extract target variables
df_train["start_month"] = df_train["parsed_start_date"].dt.month
df_train["start_day"] = df_train["parsed_start_date"].dt.day
df_train["end_month"] = df_train["parsed_end_date"].dt.month
df_train["end_day"] = df_train["parsed_end_date"].dt.day

# Vectorize headlines
vectorizer = TfidfVectorizer(max_features=1000)
X_train = vectorizer.fit_transform(df_train["headline"])

# Train models
model_sm = RandomForestClassifier().fit(X_train, df_train["start_month"])
model_sd = RandomForestClassifier().fit(X_train, df_train["start_day"])
model_em = RandomForestClassifier().fit(X_train, df_train["end_month"])
model_ed = RandomForestClassifier().fit(X_train, df_train["end_day"])


In [89]:
# === STEP 2: Load the full dataset to apply model predictions ===
df_full = pd.read_csv("heards_structured_cleaned_with_updatedDate.csv", parse_dates=["updatedDate"])
df_predict = df_full.dropna(subset=["headline", "updatedDate"]).copy()

# Predict components using TF-IDF
X_full = vectorizer.transform(df_predict["headline"])

pred_start_month = model_sm.predict(X_full)
pred_start_day = model_sd.predict(X_full)
pred_end_month = model_em.predict(X_full)
pred_end_day = model_ed.predict(X_full)

# Combine with updatedDate year
df_predict["predicted_start_date"] = pd.to_datetime({
    "year": df_predict["updatedDate"].dt.year,
    "month": pred_start_month,
    "day": pred_start_day
}, errors="coerce")

df_predict["predicted_end_date"] = pd.to_datetime({
    "year": df_predict["updatedDate"].dt.year,
    "month": pred_end_month,
    "day": pred_end_day
}, errors="coerce")


In [90]:
# === STEP 3: Write predictions back to full dataset ===
df_full.loc[df_predict.index, "start_date"] = df_predict["predicted_start_date"]
df_full.loc[df_predict.index, "end_date"] = df_predict["predicted_end_date"]

# Optional: convert to just date (drop time)
df_full["start_date"] = pd.to_datetime(df_full["start_date"]).dt.date
df_full["end_date"] = pd.to_datetime(df_full["end_date"]).dt.date

# Save final output
df_full.to_csv("heards_structured_final_with_model_dates.csv", index=False)
print("✅ Saved as 'heards_structured_final_with_model_dates.csv'")


✅ Saved as 'heards_structured_final_with_model_dates.csv'


In [91]:

# Load the specified file
df_final = pd.read_csv('heards_structured_final_with_model_dates.csv')

# Get the total number of rows
total_rows = len(df_final)

# Calculate the percentage of non-null values for each column
column_fill_percentage = (df_final.count() / total_rows) * 100

# Create a summary DataFrame
column_stats = pd.DataFrame({
    'Total Rows': total_rows,
    'Non-Null Count': df_final.notna().sum(),
    '% Filled': column_fill_percentage,
    'Data Type': df_final.dtypes
})

# Sort by fill percentage
sorted_column_stats = column_stats.sort_values(by='% Filled', ascending=False)

# Display the results
print("\n📊 Column Fill Rate & Data Types for 'heards_structured_final_with_model_dates.csv':")
print(sorted_column_stats)

# Print the number of rows
print(f"\nTotal number of rows: {total_rows}")

# Randomly print 5 rows
print("\nRandom 5 rows:")
print(df_final.sample(5).to_markdown(index=False))


📊 Column Fill Rate & Data Types for 'heards_structured_final_with_model_dates.csv':
               Total Rows  Non-Null Count    % Filled Data Type
updatedDate        195672          195672  100.000000    object
action_type        195672          195672  100.000000    object
price              195672          195672  100.000000   float64
headline           195672          195672  100.000000    object
start_date         195672          194553   99.428125    object
end_date           195672          194243   99.269696    object
party              195672          178357   91.151008    object
grade              195672          174502   89.180874   float64
location           195672          161898   82.739482    object
incoterm           195672          160850   82.203892    object
price_basis        195672          158335   80.918578    object
volume_min         195672          119534   61.088965   float64
volume_max         195672          119534   61.088965   float64
frequency_tag      

In [92]:

# Download the specified file
files.download('heards_structured_final_with_model_dates.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>